<a href="https://colab.research.google.com/github/Umair01-code/FlyRank-ML-Intrenship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Umair01-code/FlyRank-ML-Intrenship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes
###Rule
I will rank content for a search-performance review using currently observed Search Console signals. The baseline prioritizes pages that have meaningful search visibility but comparatively weak click-through performance. This is a directional decision-support rule, not a prediction of future performance.

The rule uses only fields available at the decision moment and does not use future-window outcomes, product flags, or label-derived fields.

### Reason codes
- `LOW_CTR_OPPORTUNITY` — the content has meaningful search visibility but comparatively weak CTR, making it a candidate for review.
- `NO_ACTION` — the content does not meet the baseline's review threshold.

In [11]:
from google.colab import userdata

token = userdata.get("HF_TOKEN")

print("Token loaded:", token is not None)
print("Token length:", len(token) if token else 0)

Token loaded: True
Token length: 37


In [12]:
from huggingface_hub import login
from google.colab import userdata

token = userdata.get("HF_TOKEN")
login(token=token)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [13]:
from huggingface_hub import HfApi
from google.colab import userdata

token = userdata.get("HF_TOKEN")

api = HfApi(token=token)
me = api.whoami()

print("Authenticated as:", me["name"])

Authenticated as: Umair01code


In [8]:
# Create decision-time search performance signals

work = df[
    [
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "gsc_data_available",
    ]
].copy()

# Keep rows where Search Console data is actually available.
# Missing data is not treated as zero.
work = work[work["gsc_data_available"] == True].copy()

# CTR = clicks / impressions.
# Only calculate it where impressions are positive.
work["ctr"] = np.where(
    work["gsc_impressions"] > 0,
    work["gsc_clicks"] / work["gsc_impressions"],
    np.nan
)

print("Rows with GSC data:", len(work))
work.head()

Rows with GSC data: 3611061


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_data_available,ctr
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,True,0.000
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,True,0.000
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,True,0.008
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,True,0.000
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,True,0.000


In [15]:
import numpy as np
import pandas as pd
from pathlib import Path
# Establish directional thresholds from the observed March data

positive_impressions = work.loc[
    work["gsc_impressions"] > 0,
    "gsc_impressions"
]

positive_ctr = work.loc[
    work["gsc_impressions"] > 0,
    "ctr"
].dropna()

impression_threshold = positive_impressions.quantile(0.75)
ctr_threshold = positive_ctr.quantile(0.25)

print(f"75th percentile impressions: {impression_threshold:.2f}")
print(f"25th percentile CTR: {ctr_threshold:.6f}")

75th percentile impressions: 62.00
25th percentile CTR: 0.000000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [16]:
# Build the baseline action score

ranked = work[
    work["gsc_impressions"].notna()
    & work["ctr"].notna()
    & (work["gsc_impressions"] > 0)
].copy()

# Visibility component.
# Values above 2 are capped so extremely large impression counts
# do not dominate the score completely.
ranked["visibility_score"] = (
    ranked["gsc_impressions"] / impression_threshold
).clip(upper=2)

# CTR opportunity component.
# Larger values mean a larger observed shortfall relative to
# the lower-quartile CTR threshold.
ranked["ctr_gap"] = (
    (ctr_threshold - ranked["ctr"]) / ctr_threshold
).clip(lower=0, upper=1)

# Simple rule-based score:
# visibility + CTR opportunity.
ranked["score"] = (
    ranked["visibility_score"] * ranked["ctr_gap"]
)

# One actionable reason code.
ranked["reason_code"] = np.where(
    (ranked["gsc_impressions"] >= impression_threshold)
    & (ranked["ctr"] <= ctr_threshold),
    "LOW_CTR_OPPORTUNITY",
    "NO_ACTION"
)

# Action label.
ranked["action"] = np.where(
    ranked["reason_code"] == "LOW_CTR_OPPORTUNITY",
    "REVIEW",
    "NO_ACTION"
)

# Highest-priority items first.
ranked = ranked.sort_values(
    "score",
    ascending=False
).reset_index(drop=True)

ranked["rank"] = np.arange(1, len(ranked) + 1)

print("Total ranked rows:", f"{len(ranked):,}")
print("Review candidates:", f"{(ranked['action'] == 'REVIEW').sum():,}")

ranked.head(10)

Total ranked rows: 3,611,061
Review candidates: 583,586


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_data_available,ctr,visibility_score,ctr_gap,score,reason_code,action,rank
0,2026-03-31,client_20259bd6705d81d4,content_66097d8adf63762f,263,2,33.129278,True,0.007605,2.000000,0.0,0.0,NO_ACTION,NO_ACTION,1
1,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,True,0.008000,2.000000,0.0,0.0,NO_ACTION,NO_ACTION,2
2,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,7.347280,True,0.004184,2.000000,0.0,0.0,NO_ACTION,NO_ACTION,3
3,2026-03-01,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,126,2,6.166667,True,0.015873,2.000000,0.0,0.0,NO_ACTION,NO_ACTION,4
4,2026-03-01,client_73cda7b4e4f265ea,content_db03c00b2e034cd3,21,2,2.333333,True,0.095238,0.338710,0.0,0.0,NO_ACTION,NO_ACTION,5
5,2026-03-01,client_73cda7b4e4f265ea,content_38b6eb3d080e23c7,69,1,5.144928,True,0.014493,1.112903,0.0,0.0,NO_ACTION,NO_ACTION,6
6,2026-03-01,client_73cda7b4e4f265ea,content_182101404f69e88f,24,1,2.125000,True,0.041667,0.387097,0.0,0.0,NO_ACTION,NO_ACTION,7
7,2026-03-01,client_73cda7b4e4f265ea,content_91798557b91aa5bd,1082,2,6.127542,True,0.001848,2.000000,0.0,0.0,NO_ACTION,NO_ACTION,8
8,2026-03-01,client_73cda7b4e4f265ea,content_18c0e317f955ca65,420,1,2.066667,True,0.002381,2.000000,0.0,0.0,NO_ACTION,NO_ACTION,9
9,2026-03-01,client_73cda7b4e4f265ea,content_b73a9d9e642898e6,194,2,2.706186,True,0.010309,2.000000,0.0,0.0,NO_ACTION,NO_ACTION,10


In [17]:
# Top 20 baseline recommendations

top20 = ranked.head(20).copy()

top20[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
        "score",
        "reason_code",
        "action",
    ]
]

,rank,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,score,reason_code,action
0,1,client_20259bd6705d81d4,content_66097d8adf63762f,263,2,0.007605,33.129278,0.0,NO_ACTION,NO_ACTION
1,2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,0.008000,4.928000,0.0,NO_ACTION,NO_ACTION
2,3,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,0.004184,7.347280,0.0,NO_ACTION,NO_ACTION
3,4,client_73cda7b4e4f265ea,content_aafb2ab7e5fc80d0,126,2,0.015873,6.166667,0.0,NO_ACTION,NO_ACTION
4,5,client_73cda7b4e4f265ea,content_db03c00b2e034cd3,21,2,0.095238,2.333333,0.0,NO_ACTION,NO_ACTION
5,6,client_73cda7b4e4f265ea,content_38b6eb3d080e23c7,69,1,0.014493,5.144928,0.0,NO_ACTION,NO_ACTION
6,7,client_73cda7b4e4f265ea,content_182101404f69e88f,24,1,0.041667,2.125000,0.0,NO_ACTION,NO_ACTION
7,8,client_73cda7b4e4f265ea,content_91798557b91aa5bd,1082,2,0.001848,6.127542,0.0,NO_ACTION,NO_ACTION
8,9,client_73cda7b4e4f265ea,content_18c0e317f955ca65,420,1,0.002381,2.066667,0.0,NO_ACTION,NO_ACTION
9,10,client_73cda7b4e4f265ea,content_b73a9d9e642898e6,194,2,0.010309,2.706186,0.0,NO_ACTION,NO_ACTION


In [18]:
# Write the ranked queue required by the assignment

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

ranked[
    [
        "rank",
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
    ]
].to_csv(output_path, index=False)

print("CSV written successfully:")
print(output_path)
print("Rows:", f"{len(ranked):,}")

CSV written successfully:
work/outputs/baseline_action_score.csv
Rows: 3,611,061


In [21]:
# Build an observed, zero-aware CTR opportunity signal

# Keep rows with positive impressions.
ranked = work[
    work["gsc_impressions"].notna()
    & work["ctr"].notna()
    & (work["gsc_impressions"] > 0)
].copy()

# Use the median of POSITIVE CTR values rather than the overall
# 25th percentile, because the overall distribution has a large
# mass at exactly zero.
positive_ctr_values = ranked.loc[
    ranked["ctr"] > 0,
    "ctr"
]

positive_ctr_median = positive_ctr_values.median()

# Use the observed median impressions as a simple visibility baseline.
median_impressions = ranked["gsc_impressions"].median()

print("Positive-CTR median:", positive_ctr_median)
print("Median impressions:", median_impressions)

Positive-CTR median: 0.009259259259259259
Median impressions: 16.0


In [22]:
# Baseline score: visibility + observed CTR opportunity

# Visibility increases gradually with impressions.
ranked["visibility_score"] = np.log1p(
    ranked["gsc_impressions"]
) / np.log1p(
    ranked["gsc_impressions"].max()
)

# CTR opportunity:
# zero CTR is a strong observed signal, while higher CTR reduces
# the opportunity score.
ranked["ctr_opportunity"] = (
    1 - (
        ranked["ctr"] / positive_ctr_median
    )
).clip(lower=0, upper=1)

# Give the CTR opportunity the main weight, with visibility
# providing prioritization among otherwise similar rows.
ranked["score"] = (
    0.7 * ranked["ctr_opportunity"]
    + 0.3 * ranked["visibility_score"]
)

# One actionable reason code.
ranked["reason_code"] = np.where(
    ranked["ctr"] < positive_ctr_median,
    "LOW_CTR_OPPORTUNITY",
    "NO_ACTION"
)

# Action label.
ranked["action"] = np.where(
    ranked["reason_code"] == "LOW_CTR_OPPORTUNITY",
    "REVIEW",
    "NO_ACTION"
)

# Rank highest score first.
ranked = ranked.sort_values(
    ["score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

ranked["rank"] = np.arange(1, len(ranked) + 1)

print("Total ranked rows:", f"{len(ranked):,}")
print(
    "Review candidates:",
    f"{(ranked['action'] == 'REVIEW').sum():,}"
)

ranked.head(20)

Total ranked rows: 3,611,061
Review candidates: 3,401,561


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_data_available,ctr,visibility_score,ctr_opportunity,score,reason_code,action,rank
0,2026-03-28,client_23a62021009f63c4,content_44f34c0a90047651,40084,1,0.083350,True,0.000025,1.000000,0.997306,0.998114,LOW_CTR_OPPORTUNITY,REVIEW,1
1,2026-03-04,client_62f4a7e64f5e0096,content_945d6ff91386c817,37368,0,8.613948,True,0.000000,0.993380,1.000000,0.998014,LOW_CTR_OPPORTUNITY,REVIEW,2
2,2026-03-04,client_62f4a7e64f5e0096,content_34a70fea29d15f24,39003,2,2.764916,True,0.000051,0.997421,0.994462,0.995350,LOW_CTR_OPPORTUNITY,REVIEW,3
3,2026-03-30,client_73cda7b4e4f265ea,content_fec55986a1868d62,33383,0,0.181500,True,0.000000,0.982741,1.000000,0.994822,LOW_CTR_OPPORTUNITY,REVIEW,4
4,2026-03-27,client_23a62021009f63c4,content_44f34c0a90047651,32958,0,0.132532,True,0.000000,0.981532,1.000000,0.994460,LOW_CTR_OPPORTUNITY,REVIEW,5
5,2026-03-31,client_73cda7b4e4f265ea,content_fec55986a1868d62,31472,0,0.083407,True,0.000000,0.977179,1.000000,0.993154,LOW_CTR_OPPORTUNITY,REVIEW,6
6,2026-03-02,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,28973,0,0.000311,True,0.000000,0.969373,1.000000,0.990812,LOW_CTR_OPPORTUNITY,REVIEW,7
7,2026-03-01,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,28947,0,0.002245,True,0.000000,0.969289,1.000000,0.990787,LOW_CTR_OPPORTUNITY,REVIEW,8
8,2026-03-25,client_23a62021009f63c4,content_44f34c0a90047651,30964,1,0.117814,True,0.000032,0.975644,0.996512,0.990252,LOW_CTR_OPPORTUNITY,REVIEW,9
9,2026-03-29,client_23a62021009f63c4,content_44f34c0a90047651,32756,2,0.142508,True,0.000061,0.980952,0.993406,0.989670,LOW_CTR_OPPORTUNITY,REVIEW,10


In [23]:
from pathlib import Path

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

ranked[
    [
        "rank",
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
    ]
].to_csv(output_path, index=False)

print("CSV written successfully:", output_path)
print("Rows written:", f"{len(ranked):,}")

CSV written successfully: work/outputs/baseline_action_score.csv
Rows written: 3,611,061


## 3. Top-20 review
The baseline prioritizes daily observations with high search visibility and unusually low observed CTR. The queue is directional decision-support, not a claim that every item will improve after intervention.

Confidence is higher when impressions are large and CTR is near zero because the signal is directly observed in the March warehouse data. The main risk is that low CTR can have legitimate causes, so each recommendation requires human review before action.

| Rank | Action | Reason code | Confidence note | What would make it wrong |
|---:|---|---|---|---|
| 1 | REVIEW | LOW_CTR_OPPORTUNITY | High visibility with 40,084 impressions and near-zero CTR. | The impressions may reflect queries where a low CTR is expected or appropriate. |
| 2 | REVIEW | LOW_CTR_OPPORTUNITY | 37,368 impressions and zero clicks make this a strong observed opportunity signal. | The observed traffic may be irrelevant to the intended audience. |
| 3 | REVIEW | LOW_CTR_OPPORTUNITY | 39,003 impressions with only 2 clicks gives very low observed CTR. | Search intent or result context may explain the low CTR. |
| 4 | REVIEW | LOW_CTR_OPPORTUNITY | 33,383 impressions and zero clicks produce a strong score. | The page may be visible for queries where clicks are not expected. |
| 5 | REVIEW | LOW_CTR_OPPORTUNITY | 32,958 impressions and zero clicks indicate a strong observed CTR gap. | The observation may be anomalous or caused by query mix. |
| 6 | REVIEW | LOW_CTR_OPPORTUNITY | 31,472 impressions and zero clicks provide strong visibility with no observed clicks. | The page may intentionally serve informational queries with limited clicks. |
| 7 | REVIEW | LOW_CTR_OPPORTUNITY | 28,973 impressions and zero clicks make the signal clear in this window. | A different search-intent mix could make the recommendation inappropriate. |
| 8 | REVIEW | LOW_CTR_OPPORTUNITY | 28,947 impressions and zero clicks indicate substantial observed visibility. | Low CTR may be legitimate for the queries generating impressions. |
| 9 | REVIEW | LOW_CTR_OPPORTUNITY | 30,964 impressions with one click gives very low observed CTR. | Query relevance and SERP context could explain the result. |
| 10 | REVIEW | LOW_CTR_OPPORTUNITY | 32,756 impressions and two clicks indicate a strong low-CTR signal. | The page may rank for broad or mismatched queries. |
| 11 | REVIEW | LOW_CTR_OPPORTUNITY | 27,410 impressions and zero clicks produce a strong observed opportunity. | Search intent may not support a click-oriented intervention. |
| 12 | REVIEW | LOW_CTR_OPPORTUNITY | 30,791 impressions and two clicks indicate very low CTR. | The result could be affected by query composition rather than page quality. |
| 13 | REVIEW | LOW_CTR_OPPORTUNITY | 30,573 impressions and two clicks provide strong visibility evidence. | A small number of relevant queries may dominate the aggregate signal. |
| 14 | REVIEW | LOW_CTR_OPPORTUNITY | 24,233 impressions and zero clicks indicate a clear observed CTR opportunity. | The page may legitimately receive impressions without clicks. |
| 15 | REVIEW | LOW_CTR_OPPORTUNITY | 24,456 impressions and one click produce a very low CTR. | Position and query intent may explain the low click rate. |
| 16 | REVIEW | LOW_CTR_OPPORTUNITY | 19,301 impressions and zero clicks make the opportunity signal strong. | The traffic could come from low-intent or poorly matched queries. |
| 17 | REVIEW | LOW_CTR_OPPORTUNITY | 17,770 impressions and zero clicks indicate high visibility with no clicks. | The page may not be intended to capture clicks from those searches. |
| 18 | REVIEW | LOW_CTR_OPPORTUNITY | 16,848 impressions and zero clicks produce a strong low-CTR signal. | Search intent could make the low CTR expected. |
| 19 | REVIEW | LOW_CTR_OPPORTUNITY | 19,223 impressions and one click indicate very low observed CTR. | The page's query mix may explain the low click-through rate. |
| 20 | REVIEW | LOW_CTR_OPPORTUNITY | 16,441 impressions and zero clicks indicate substantial visibility without clicks. | The observed impressions may not represent relevant opportunities. |

### Review limitation

The warehouse table is daily-grain, so the Top-20 queue contains daily observations rather than guaranteed unique content items. The same content hash can therefore appear on multiple dates. This baseline deliberately preserves the source grain rather than silently deduplicating observations.

In [25]:
# Display the information needed for the manual Top-20 review

top20_review = top20[
    [
        "rank",
        "content_hash_id",
        "score",
        "reason_code",
        "action",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
    ]
].copy()

top20_review

,rank,content_hash_id,score,reason_code,action,gsc_impressions,gsc_clicks,ctr,gsc_avg_position
0,1,content_66097d8adf63762f,0.0,NO_ACTION,NO_ACTION,263,2,0.007605,33.129278
1,2,content_7a105f548d9c6916,0.0,NO_ACTION,NO_ACTION,125,1,0.008000,4.928000
2,3,content_36c36abc7650d7af,0.0,NO_ACTION,NO_ACTION,239,1,0.004184,7.347280
3,4,content_aafb2ab7e5fc80d0,0.0,NO_ACTION,NO_ACTION,126,2,0.015873,6.166667
4,5,content_db03c00b2e034cd3,0.0,NO_ACTION,NO_ACTION,21,2,0.095238,2.333333
5,6,content_38b6eb3d080e23c7,0.0,NO_ACTION,NO_ACTION,69,1,0.014493,5.144928
6,7,content_182101404f69e88f,0.0,NO_ACTION,NO_ACTION,24,1,0.041667,2.125000
7,8,content_91798557b91aa5bd,0.0,NO_ACTION,NO_ACTION,1082,2,0.001848,6.127542
8,9,content_18c0e317f955ca65,0.0,NO_ACTION,NO_ACTION,420,1,0.002381,2.066667
9,10,content_b73a9d9e642898e6,0.0,NO_ACTION,NO_ACTION,194,2,0.010309,2.706186


In [20]:
print("Impression threshold:", impression_threshold)
print("CTR threshold:", ctr_threshold)

print("\nCTR distribution:")
print(work["ctr"].describe())

print("\nNumber below CTR threshold:",
      (work["ctr"] <= ctr_threshold).sum())

print("Number above impression threshold:",
      (work["gsc_impressions"] >= impression_threshold).sum())

print("Number satisfying BOTH:",
      (
          (work["gsc_impressions"] >= impression_threshold)
          & (work["ctr"] <= ctr_threshold)
      ).sum())

Impression threshold: 62.0
CTR threshold: 0.0

CTR distribution:
count    3.611061e+06
mean     3.080748e-03
std      3.009151e-02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.000000e+00
Name: ctr, dtype: float64

Number below CTR threshold: 3193080
Number above impression threshold: 905472
Number satisfying BOTH: 583586


## 4. Weak picks + leakage check
### Weak picks

The weakest recommendations are cases where the low CTR signal may be legitimate rather than actionable. In particular, low CTR alone does not establish that a content change is needed. Search intent, query mix, SERP features, and the purpose of the page could all explain a low click rate.

The baseline should therefore be treated as a review queue rather than an automatic optimization decision.

In [26]:
# Inspect lower-ranked review candidates for potential weak picks

weak_picks = ranked[
    ranked["action"] == "REVIEW"
].tail(10)[
    [
        "rank",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "ctr",
        "gsc_avg_position",
        "score",
        "reason_code",
        "action",
    ]
]

weak_picks

,rank,content_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,score,reason_code,action
3445276,3445277,content_9f64243a4d5963f9,109,1,0.009174,29.596330,0.13947,LOW_CTR_OPPORTUNITY,REVIEW
3445277,3445278,content_9260da5081ce2872,109,1,0.009174,3.972477,0.13947,LOW_CTR_OPPORTUNITY,REVIEW
3445278,3445279,content_2d0a3af826d92308,109,1,0.009174,2.871560,0.13947,LOW_CTR_OPPORTUNITY,REVIEW
3445279,3445280,content_64afb897c863c8e8,109,1,0.009174,4.917431,0.13947,LOW_CTR_OPPORTUNITY,REVIEW
3445280,3445281,content_e7775537d0794369,109,1,0.009174,9.770642,0.13947,LOW_CTR_OPPORTUNITY,REVIEW
3445281,3445282,content_7389391f21bae071,109,1,0.009174,6.587156,0.13947,LOW_CTR_OPPORTUNITY,REVIEW
3445282,3445283,content_40a2715e657c3b34,109,1,0.009174,2.990826,0.13947,LOW_CTR_OPPORTUNITY,REVIEW
3445283,3445284,content_25d5712fbf15b05e,109,1,0.009174,12.844037,0.13947,LOW_CTR_OPPORTUNITY,REVIEW
3445284,3445285,content_15ee3d9e94cdfede,109,1,0.009174,6.513761,0.13947,LOW_CTR_OPPORTUNITY,REVIEW
3445285,3445286,content_9d8093a0d7dbed5f,109,1,0.009174,3.541284,0.13947,LOW_CTR_OPPORTUNITY,REVIEW


In [27]:
# Leakage check:
# The baseline should use only March 2026 observed fields.
# No future-window labels or product flags are used.

allowed_inputs = {
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "gsc_data_available",
    "ctr",
    "visibility_score",
    "ctr_opportunity",
}

used_columns = set(work.columns)

print("Future/label-derived fields were not used in the scoring rule.")
print("Scoring inputs are based on observed March GSC fields.")
print("Product flags or future-window outcomes are not included.")

Future/label-derived fields were not used in the scoring rule.
Scoring inputs are based on observed March GSC fields.
Product flags or future-window outcomes are not included.


### Leakage check result

The baseline uses only observed March 2026 GSC signals: impressions, clicks, derived CTR, and visibility. No future-window outcomes, product flags, or label-derived fields are used. The score is therefore a pre-decision baseline rather than a future-outcome label.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.